# Final Data Preparation and Load Prep

This notebook serves as the final step in our data processing pipeline. Here, we consolidate the insights gained from Exploratory Data Analysis and Statistical Analysis into final, structured datasets ready for loading into a database or BI tool.

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os

import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


## 1. Load Cleaned Data

In [2]:
# Path to cleaned data
data_path = '../data/processed/cleaned_data.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
    print(f"Successfully loaded {len(df):,} records.")
else:
    print("Error: cleaned_data.csv not found. Please run the cleaning notebook first.")

Successfully loaded 392,732 records.


## 2. Customer-Level Data Consolidation
We will aggregate data at the customer level and re-attach the RFM segments and Clusters identified in the statistical analysis.

In [3]:
# Define reference date for recency calculation
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)

# Aggregate metrics at CustomerID level
customer_df = df.dropna(subset=['CustomerID']).groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum',
    'Country': 'first'
}).reset_index()

# Rename columns for clarity
customer_df.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'Revenue': 'Monetary'
}, inplace=True)

print(f"Created customer-level dataset with {len(customer_df)} unique customers.")

Created customer-level dataset with 4339 unique customers.


### 2.1 Re-calculating Clusters (K-Means)
We use the same parameters as in the Statistical Analysis notebook (K=4).

In [4]:
# Preprocessing for K-Means
rfm_log = customer_df[['Recency', 'Frequency', 'Monetary']].apply(lambda x: np.log(x + 1))
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Apply K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_df['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Add RFM Scores (1-4 labels)
customer_df['R_Score'] = pd.qcut(customer_df['Recency'], 4, labels=[4, 3, 2, 1])
customer_df['F_Score'] = pd.qcut(customer_df['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
customer_df['M_Score'] = pd.qcut(customer_df['Monetary'], 4, labels=[1, 2, 3, 4])

# Total RFM Score
customer_df['RFM_Segment'] = customer_df['R_Score'].astype(str) + customer_df['F_Score'].astype(str) + customer_df['M_Score'].astype(str)

customer_df.head()

,CustomerID,Recency,Frequency,Monetary,Country,Cluster,R_Score,F_Score,M_Score,RFM_Segment
0,12346,326,1,77183.60,United Kingdom,1,1,1,4,114
1,12347,2,7,4310.00,Iceland,2,4,4,4,444
2,12348,75,4,1797.24,Finland,1,2,3,4,234
3,12349,19,1,1757.55,Italy,0,3,1,4,314
4,12350,310,1,334.40,Norway,3,1,1,2,112


## 3. Product-Level Data Consolidation
Aggregating sales metrics by StockCode.

In [5]:
product_df = df.groupby(['StockCode', 'Description']).agg({
    'Revenue': 'sum',
    'Quantity': 'sum',
    'InvoiceNo': 'nunique',
    'CustomerID': 'nunique'
}).reset_index()

product_df.rename(columns={
    'Revenue': 'TotalRevenue',
    'Quantity': 'TotalQuantity',
    'InvoiceNo': 'UniqueInvoices',
    'CustomerID': 'UniqueCustomers'
}, inplace=True)

print(f"Created product-level dataset with {len(product_df)} unique products.")

Created product-level dataset with 3897 unique products.


## 4. Export Final Datasets
We save the consolidated data into the `processed` directory for final loading.

In [6]:
# Define output paths
customer_output = '../data/processed/final_customer_data.csv'
product_output = '../data/processed/final_product_data.csv'

# Export to CSV
customer_df.to_csv(customer_output, index=False)
product_df.to_csv(product_output, index=False)

print(f"Successfully exported customer data to: {customer_output}")
print(f"Successfully exported product data to: {product_output}")

Successfully exported customer data to: ../data/processed/final_customer_data.csv
Successfully exported product data to: ../data/processed/final_product_data.csv


## 5. Summary and Next Steps

The data is now prepared in two primary dimensions:
1. **Customer Dimension**: Includes behavior metrics (RFM) and segmentation (Clusters).
2. **Product Dimension**: Includes sales performance and market reach metrics.

**Suggested Next Steps:**
- Load `final_customer_data.csv` into a visualization tool like Tableau or PowerBI for CRM dashboards.
- Load `final_product_data.csv` for inventory optimization analysis.
- Perform predictive modeling (e.g., Churn Prediction) using the customer-level features.